# Pipeline
A pipeline sequences a multiple tasks together. e.g. When you process an input through LLM, you perform the following tasks:

Raw_text --> Tokenize (generate token ids) --> MODEL (process the tokens) --> logits --> Prediction (decode the logits into text)

These tasks can be performed individually or you can use the HF pipeline which does them all for you in one go.

### Without Pipeline

In [12]:
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)


In [6]:
# tokenize a batch of texts
raw_inputs = ["I love using transformers library!", "This is the worst movie I have ever seen."]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print(f"Tokenized inputs: {inputs}")

Tokenized inputs: {'input_ids': tensor([[  101,  1045,  2293,  2478, 19081,  3075,   999,   102,     0,     0,
             0,     0],
        [  101,  2023,  2003,  1996,  5409,  3185,  1045,  2031,  2412,  2464,
          1012,   102]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


In [13]:
# load the pre-trained model
# model = AutoModel.from_pretrained(checkpoint, num_labels=2)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

print(f"Model architecture: {model}")

Model architecture: DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=

In [17]:
# predict sentiment labels
outputs = model(**inputs)
# print(f"Logits: {outputs.last_hidden_state.shape}") # works with AutoModel
# shape: (batch_size, sequence_length, hidden_size)
# (2, 16, 768) for DistilBERT

print(f"Logits Shape: {outputs.logits.shape}")  # works with AutoModelForSequenceClassification
# shape: (batch_size, num_labels)
# (2, 2) for DistilBERT fine-tuned on SST-2
print(f"Logits: {outputs.logits}")

Logits Shape: torch.Size([2, 2])
Logits: tensor([[-3.3502,  3.4931],
        [ 4.6341, -3.6792]], grad_fn=<AddmmBackward0>)


In [25]:
import torch

# Decoding the predicted labels
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1)



# model.config.id2label(predictions.argmax(dim=-1).item())  # get the label with the highest score
for i, prediction in enumerate(predictions):
    predicted_label = model.config.id2label[prediction.argmax(dim=-1).item()]
    confidence = prediction.max().item()
    print(f"Input Text: {raw_inputs[i]}")
    print(f"Predicted Label: {predicted_label}, Confidence: {confidence:.4f}\n")

Input Text: I love using transformers library!
Predicted Label: POSITIVE, Confidence: 0.9989

Input Text: This is the worst movie I have ever seen.
Predicted Label: NEGATIVE, Confidence: 0.9998



### With Pipeline

In [27]:
from transformers import pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
results = sentiment_pipeline(raw_inputs)
for i, result in enumerate(results):
    print(f"Input Text: {raw_inputs[i]}")
    print(f"Predicted Label: {result['label']}, Confidence: {result['score']:.4f}\n")

Device set to use cpu


Input Text: I love using transformers library!
Predicted Label: POSITIVE, Confidence: 0.9989

Input Text: This is the worst movie I have ever seen.
Predicted Label: NEGATIVE, Confidence: 0.9998

